# 🤖 Machine Learning — Notebook de Projeto

**Disciplina:** Machine Learning  
**Professor:** Messias Batista  
**Aluno(a):**  
**Data:**  
**Dataset:** MNIST 784 (OpenML)  
**Problema de negócio:** Classificação de dígitos manuscritos (0–9) a partir de imagens 28×28 pixels, comparando o desempenho de diferentes algoritmos de ensemble learning.  

---
## 1. 📦 Importações

Importe aqui todas as bibliotecas que serão utilizadas ao longo do projeto.

Você precisará de bibliotecas para:
- **Manipulação de dados** — leitura, transformação e análise de tabelas
- **Visualização** — criação de gráficos e figuras
- **Pré-processamento** — divisão dos dados, normalização e codificação de variáveis
- **Algoritmos de Machine Learning** — os modelos que serão treinados
- **Métricas de avaliação** — para medir o desempenho dos modelos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import (
    RandomForestClassifier,
    BaggingClassifier,
    AdaBoostClassifier,
    VotingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

---
## 2. 📂 Carregamento dos Dados

Carregue o dataset e faça uma primeira inspeção para entender com o que você está trabalhando.

Nesta etapa você deve responder:
- Quantas linhas e colunas o dataset possui?
- Quais são os tipos de cada coluna?
- Existem valores nulos? Em quais colunas e em qual quantidade?

In [ ]:
# Carregando dados de https://www.openml.org/d/554
X, y = fetch_openml('mnist_784', version=1, return_X_y=True)

In [ ]:
print(f"Dimensões: {X.shape}")
print(f"\nTipos de dados:\n{X.dtypes.value_counts()}")
print(f"\nTotal de valores nulos: {X.isnull().sum().sum()}")
print(f"\nClasses em y: {sorted(y.unique())}")

---
## 3. 🔍 Análise Exploratória de Dados (EDA)

Explore os dados antes de construir qualquer modelo. Esta é uma das etapas mais importantes do processo.

Nesta etapa você deve:
- Calcular estatísticas descritivas das variáveis numéricas
- Analisar a distribuição da variável alvo (está balanceada?)
- Visualizar a distribuição das demais variáveis
- Identificar possíveis outliers
- Analisar a correlação entre as variáveis

In [ ]:
# Estatísticas descritivas
X.describe()

In [ ]:
# Distribuição da variável alvo
ax = y.value_counts().sort_index().plot(
    kind='bar', figsize=(10, 4), title='Distribuição das classes (dígitos)'
)
ax.set_xlabel('Dígito')
ax.set_ylabel('Quantidade')
plt.tight_layout()
plt.show()

In [ ]:
# Visualização de um exemplo de cada dígito
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for digit in range(10):
    idx = (y == str(digit)).idxmax()
    ax = axes[digit // 5][digit % 5]
    ax.imshow(X.loc[idx].values.reshape(28, 28), cmap='gray')
    ax.set_title(f'Dígito: {digit}')
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## 4. 🛠️ Pré-processamento

Prepare os dados para que o modelo consiga aprender corretamente.

Nesta etapa você deve:
- Remover colunas que não contribuem para o modelo
- Tratar os valores nulos (remover ou preencher)
- Converter variáveis categóricas em numéricas
- Separar as features (X) da variável alvo (y)
- Aplicar normalização ou padronização se necessário

In [ ]:
# Redução da base para fins didáticos (usando 40% dos dados)
x_temp, _, y_temp, _ = train_test_split(
    X, y, test_size=0.6, stratify=y, random_state=42
)
X = x_temp
y = y_temp

print(f"Base reduzida para: {X.shape}")

In [ ]:
# X e y já foram separados no carregamento via fetch_openml(return_X_y=True)
print(f"Features (X): {X.shape}")
print(f"Target  (y):  {y.shape}")
print(f"Classes: {sorted(y.unique())}")

---
## 5. ✂️ Separação Treino / Teste

Divida os dados em dois conjuntos: um para treinar o modelo e outro para testá-lo.

Lembre-se:
- O modelo deve ser treinado **apenas** com os dados de treino
- Os dados de teste simulam situações novas, que o modelo nunca viu
- Uma divisão comum é 70% treino e 30% teste
- Use o parâmetro `stratify` para manter a proporção das classes

In [ ]:
# Divisão treino/teste
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=4
)

print(f"Treino: {x_train.shape}")
print(f"Teste:  {x_test.shape}")

---
## 6. 🧠 Treinamento do Modelo

Escolha um algoritmo, instancie o modelo e treine-o com os dados de treino.

Lembre-se:
- O treinamento acontece com o método `fit()`
- As previsões são feitas com o método `predict()`
- Você pode testar mais de um algoritmo e compará-los na seção 8

In [ ]:
# Modelo principal: Random Forest
rf = RandomForestClassifier(n_estimators=1000, random_state=42)
rf.fit(x_train, y_train)

In [ ]:
# Previsões para o conjunto de teste
y_pred = rf.predict(x_test)

---
## 7. 📊 Avaliação do Modelo

Meça o desempenho do modelo utilizando métricas adequadas ao problema.

Para problemas de **classificação**, avalie:
- **Acurácia** — proporção de acertos em relação ao total
- **Matriz de Confusão** — visualização detalhada dos acertos e erros por classe
- **Relatório de Classificação** — precision, recall e f1-score por classe
- **Validação Cruzada** — para uma estimativa mais robusta e confiável do desempenho

In [ ]:
# Acurácia
print(f"Acurácia (Random Forest): {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão — Random Forest')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

In [ ]:
# Relatório de Classificação
print(classification_report(y_test, y_pred))

In [ ]:
# Validação Cruzada
scores = cross_val_score(rf, x_train, y_train, scoring='accuracy', cv=5)
print(f"Scores: {scores}")
print(f"Média: {scores.mean():.4f} ± {scores.std():.4f}")

---
## 8. 🏆 Comparação de Modelos

Teste outros algoritmos e compare os resultados para identificar o mais adequado ao seu problema.

Dica: use validação cruzada para comparar os modelos de forma justa,
pois ela elimina o efeito da aleatoriedade de uma única divisão treino/teste.

In [ ]:
results = pd.DataFrame()

# Random Forest (já treinado)
results['RandomForest'] = cross_val_score(rf, x_train, y_train, scoring='accuracy')

# Bagging
bg = BaggingClassifier(n_estimators=150, random_state=42)
bg.fit(x_train, y_train)
results['Bagging'] = cross_val_score(bg, x_train, y_train, scoring='accuracy')

# AdaBoost
adb = AdaBoostClassifier()
adb.fit(x_train, y_train)
results['AdaBoost'] = cross_val_score(adb, x_train, y_train, scoring='accuracy')

# Decision Tree
dc = DecisionTreeClassifier()
dc.fit(x_train, y_train)
results['DecisionTree'] = cross_val_score(dc, x_train, y_train, scoring='accuracy')

# SVC
svc = SVC()
svc.fit(x_train, y_train)
results['SVC'] = cross_val_score(svc, x_train, y_train, scoring='accuracy')

# VotingClassifier (ensemble customizado)
dt_model  = DecisionTreeClassifier()
rf_model  = RandomForestClassifier(n_estimators=20)
bg_model  = BaggingClassifier(max_samples=0.5, max_features=1.0, n_estimators=20)
adb_model = AdaBoostClassifier(n_estimators=5, learning_rate=1)

vc = VotingClassifier(
    estimators=[
        ('dt',  dt_model),
        ('rf',  rf_model),
        ('bg',  bg_model),
        ('adb', adb_model),
    ],
    voting='hard',
)
vc.fit(x_train, y_train)
y_pred_vc = vc.predict(x_test)
print(f"VotingClassifier Acurácia: {accuracy_score(y_test, y_pred_vc):.4f}")

# Boxplot comparativo
plt.figure(figsize=(14, 6))
sns.boxplot(data=results)
plt.title('Comparação de Modelos — Cross-Validation Accuracy')
plt.ylabel('Acurácia')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

results

---
## 9. 📝 Conclusões

Responda as perguntas abaixo com base nos resultados obtidos:

**1. Qual algoritmo apresentou melhor desempenho? Por quê?**  
> _Escreva aqui_

**2. O modelo está sofrendo overfitting ou underfitting? Como você identificou?**  
> _Escreva aqui_

**3. Os resultados respondem à pergunta de negócio levantada no início?**  
> _Escreva aqui_

**4. Qual seria o próximo passo para melhorar o modelo?**  
> _Escreva aqui_

---
### 🔖 Referências
- Dataset: MNIST 784 — https://www.openml.org/d/554
- Scikit-learn: https://scikit-learn.org
- Material da disciplina: www.mrafaelbatista.dev
